<a href="https://colab.research.google.com/github/kuds/mesozoic-labs/blob/main/notebooks/sb3_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Dinosaur Training Notebook

Train one of the implemented species through balance, locomotion, and a species-specific simulator task with PPO or SAC.

Run the full Stage 1 → Stage 2 → Stage 3 curriculum in order within one notebook session. Later stages consume model and result objects held in memory; files saved to Drive support analysis and archival, but this SB3 notebook does not reconstruct a cross-session curriculum continuation.

Change `SPECIES` in the configuration cell below. Stage budgets, environment settings, and algorithm hyperparameters are loaded from `configs/<species>/stage*.toml`; those files and the generated species catalog are authoritative. Values differ by species and stage, so this notebook does not duplicate them.

For a smoke test, override the budget deliberately. For a full run, start with the configured values, measure memory and throughput on the target machine, and adjust parallel environments only from observed resource use. The repository does not publish a validated hardware-to-runtime or batch-size table.

## 1. Setup & Installation

In [ ]:
# Install dependencies (Colab auto-detected; no-op locally)
import importlib
import os

IN_COLAB = "COLAB_GPU" in os.environ or "COLAB_RELEASE_TAG" in os.environ or os.path.exists("/content")

if IN_COLAB:
    # Configure headless rendering for MuJoCo (must happen before mujoco import)
    os.environ["MUJOCO_GL"] = "egl"
    NVIDIA_ICD_CONFIG_PATH = "/usr/share/glvnd/egl_vendor.d/10_nvidia.json"
    if not os.path.exists(NVIDIA_ICD_CONFIG_PATH):
        os.makedirs(os.path.dirname(NVIDIA_ICD_CONFIG_PATH), exist_ok=True)
        with open(NVIDIA_ICD_CONFIG_PATH, "w") as f:
            f.write('{"file_format_version":"1.0.0","ICD":{"library_path":"libEGL_nvidia.so.0"}}')

    # Install packages only if not already present
    if importlib.util.find_spec("mujoco") is None:
        get_ipython().system(
            'pip install -q mujoco==3.10.0 gymnasium>=0.29.0 "stable-baselines3[extra]>=2.2.0" mediapy matplotlib'
        )
    import pathlib
    import subprocess

    repo_dir = pathlib.Path("/content/mesozoic-labs")
    if not repo_dir.exists():
        subprocess.run(["git", "clone", "https://github.com/kuds/mesozoic-labs.git", str(repo_dir)], check=True)
    if importlib.util.find_spec("environments") is None:
        get_ipython().system("pip install -q -e /content/mesozoic-labs")
    print("Colab setup complete (EGL rendering enabled).")
else:
    print("Running locally.")

In [ ]:
import os
import sys
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

# Add repo root to path (works both locally from notebooks/ and in Colab)
if IN_COLAB:
    repo_root = Path("/content/mesozoic-labs")
else:
    repo_root = Path("..").resolve()

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import gymnasium as gym
import mujoco

from environments.shared.config import load_all_stages, save_stage_config
from environments.shared.curriculum import (
    RewardRampCallback,
    StageWarmupCallback,
    load_vecnorm_stats,
)
from environments.shared.evaluation import evaluate

# Library imports — shared training infrastructure
from environments.shared.train_base import (
    SpeciesConfig,
    create_vec_env,
)

print(f"MuJoCo version: {mujoco.__version__}")
print(f"Gymnasium version: {gym.__version__}")
print(f"Repo root: {repo_root}")

## 2. Configuration

Change `SPECIES` below to train a different species. All other settings apply to every species.

In [ ]:
# ===== SPECIES SELECTION =====
# Change this to: "velociraptor", "trex", "brachiosaurus", or "dibothrosuchus"
SPECIES = "velociraptor"

# ===== Training settings =====
ALGORITHM = "ppo"  # "ppo" or "sac"
N_ENVS = 4  # Number of parallel environments
SEED = 42  # Random seed for reproducibility
VERBOSE = 0  # 0=quiet, 1=progress bar, 2=debug
QUICK_TEST = False  # True = tiny run to verify setup works
USE_GOOGLE_DRIVE = True  # Mount Google Drive to save results
AUTO_DISCONNECT = True  # Disconnect the Colab runtime when training halts (gate failure or completion)

print(f"Species: {SPECIES}")
print(f"Algorithm: {ALGORITHM.upper()}")
print(f"Quick test mode: {QUICK_TEST}")

In [ ]:
# ============================================================
# Storage Configuration
# ============================================================
# When USE_GOOGLE_DRIVE is True and running in Colab, logs and
# models are saved to Google Drive so they persist across sessions.
# Otherwise, everything is saved to the local filesystem.

if USE_GOOGLE_DRIVE and IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    LOG_BASE = Path("/content/drive/MyDrive/mesozoic-labs/logs")
    LOG_BASE.mkdir(parents=True, exist_ok=True)
    print(f"Google Drive mounted. Logs will be saved to: {LOG_BASE}")
elif USE_GOOGLE_DRIVE and not IN_COLAB:
    print("Warning: USE_GOOGLE_DRIVE is True but not running in Colab. Using local storage.")
    LOG_BASE = repo_root / "logs"
    print(f"Logs will be saved to: {LOG_BASE}")
else:
    LOG_BASE = repo_root / "logs"
    print(f"Logs will be saved to: {LOG_BASE}")

# Create one run directory for the current in-memory curriculum. Re-running this
# cell in the same runtime reuses it; this is not cross-session stage continuation.
RUN_ID = ""
if not RUN_ID:
    RUN_ID = globals().get("_ACTIVE_RUN_ID", "")
if not RUN_ID:
    RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
_ACTIVE_RUN_ID = RUN_ID
RUN_DIR = LOG_BASE / SPECIES / ALGORITHM.lower() / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=True)

# Capture immutable code, runtime, seed, and plant provenance before training.
# Re-running this cell is safe when the identifying settings are unchanged.
from environments.shared.plant_contract import current_plant_identity
from environments.shared.result_bundle import initialize_result_bundle, validate_result_bundle

CHECKPOINT_SELECTION_SEED = SEED + 1000
EVALUATION_SEED = SEED + 3000
HARDWARE_LABEL = "Google Colab" if IN_COLAB else "local"
PLANT_IDENTITY = current_plant_identity(SPECIES)
PROVENANCE_PATH = initialize_result_bundle(
    RUN_DIR,
    species=SPECIES,
    algorithm=ALGORITHM,
    backend="stable-baselines3",
    seed=SEED,
    evaluation_seeds=[CHECKPOINT_SELECTION_SEED, EVALUATION_SEED],
    seed_roles={
        "training": SEED,
        "checkpoint_selection_evaluation": CHECKPOINT_SELECTION_SEED,
        "publication_evaluation": EVALUATION_SEED,
    },
    evaluation_episodes=30,
    parallel_envs=N_ENVS,
    hardware=HARDWARE_LABEL,
    plant_identity=PLANT_IDENTITY.to_dict(),
    run_id=RUN_ID,
    repository_root=repo_root,
)
print(f"Run directory: {RUN_DIR}")
print(f"Provenance:    {PROVENANCE_PATH}")

## 3. Explore the Environment

In [ ]:
# Resolve species environment class and build SpeciesConfig
import importlib

_SPECIES_MAP = {
    "velociraptor": {
        "module": "environments.velociraptor.envs.raptor_env",
        "class": "RaptorEnv",
        "stage_descriptions": "1=balance, 2=locomotion, 3=strike",
        "height_label": "Pelvis height",
        "stage3_section_label": "Hunting",
        "success_keys": ["strike_success", "bite_success"],
    },
    "trex": {
        "module": "environments.trex.envs.trex_env",
        "class": "TRexEnv",
        "stage_descriptions": "1=balance, 2=locomotion, 3=bite",
        "height_label": "Pelvis height",
        "stage3_section_label": "Hunting",
        "success_keys": ["bite_success", "strike_success"],
    },
    "brachiosaurus": {
        "module": "environments.brachiosaurus.envs.brachio_env",
        "class": "BrachioEnv",
        "stage_descriptions": "1=balance, 2=locomotion, 3=food_reach",
        "height_label": "Torso height",
        "stage3_section_label": "Food Reaching",
        "success_keys": ["food_reached"],
    },
    "dibothrosuchus": {
        "module": "environments.dibothrosuchus.envs.dibothrosuchus_env",
        "class": "DibothrosuchusEnv",
        "stage_descriptions": "1=balance, 2=locomotion, 3=snap",
        "height_label": "Trunk height",
        "stage3_section_label": "Hunting",
        "success_keys": ["snap_success"],
    },
}

assert SPECIES in _SPECIES_MAP, f"Unknown species: {SPECIES}. Choose from: {list(_SPECIES_MAP.keys())}"
_info = _SPECIES_MAP[SPECIES]
_mod = importlib.import_module(_info["module"])
EnvClass = getattr(_mod, _info["class"])

# Build SpeciesConfig (same structure used by the CLI train scripts)
SPECIES_CFG = SpeciesConfig(
    species=SPECIES,
    env_class=EnvClass,
    stage_descriptions=_info["stage_descriptions"],
    height_label=_info["height_label"],
    stage3_section_label=_info["stage3_section_label"],
    success_keys=_info["success_keys"],
)

# Load all stage configs from TOML files
STAGE_CONFIGS = load_all_stages(SPECIES)

env = EnvClass()
print(f"Environment: {EnvClass.__name__}")
print(f"Observation space: {env.observation_space.shape}")
print(f"Action space: {env.action_space.shape}")
print(f"Number of actuators: {env.model.nu}")
for stage_num, cfg in STAGE_CONFIGS.items():
    print(f"  Stage {stage_num}: {cfg['name']} - {cfg['description']}")
env.close()

In [ ]:
# Run random episodes to establish a baseline
env = EnvClass()
n_episodes = 5
episode_rewards = []
episode_lengths = []

for ep in range(n_episodes):
    obs, info = env.reset(seed=ep)
    total_reward = 0
    step = 0
    while True:
        action = env.action_space.sample()
        obs, reward, terminated, truncated, info = env.step(action)
        total_reward += reward
        step += 1
        if terminated or truncated:
            break
    episode_rewards.append(total_reward)
    episode_lengths.append(step)
    print(f"  Episode {ep + 1}: reward={total_reward:.2f}, length={step}")

print("\nRandom policy baseline:")
print(f"  Avg reward: {np.mean(episode_rewards):.2f} +/- {np.std(episode_rewards):.2f}")
print(f"  Avg length: {np.mean(episode_lengths):.1f} +/- {np.std(episode_lengths):.1f}")
env.close()

### 3b. Pre-flight: zero-action baseline

The cell above scores a **random** policy, which any trained policy beats trivially. The
floor that actually decides whether stage 1 learned anything is the **do-nothing** policy.

Actions are residuals around each species' home keyframe (`home-keyframe-residual/v1`), so
`action = 0` commands the nominal stance exactly. On a passively stable plant that is a real
policy, and on a balance stage it can be a strong one — strong enough that T-Rex runs
`20260723_204941` and `20260724_140441` both scored *below* it and advanced anyway, because
the gate at the time was `min_avg_reward = 100.0`.

Two numbers come out, and they answer different questions:

| number | meaning |
|---|---|
| **reward** | unconditional mean, including episodes the statue falls out of. A gate below this is cleared by doing nothing. |
| **reward standing** | conditioned on reaching the horizon. A gate between the two is cleared by a policy that has learned only "do not fall". |

`configs/trex/stage1_balance.toml` asks for exactly this check: *"this is a measured constant
and will go stale if the plant or the reward weights change — re-run the baseline script and
update it, or better, wire that script in as an automatic pre-stage check."* This cell is that
check. Results are written to Drive so the calibration travels with the run.

By default this measures only the species this notebook is training (~1 min) and writes the
record to that species' own log directory:

```
<LOG_BASE>/<species>/zero_action_baselines/<timestamp>.json   # the numbers, machine-readable
<LOG_BASE>/<species>/zero_action_baselines/<timestamp>.txt    # the table printed below
<RUN_DIR>/zero_action_baseline.json                           # copy, so the run carries its calibration
```

The JSON records the plant identity and the stage-1 env kwargs alongside the numbers, because
those are what make the constant go stale. Widen `BASELINE_SPECIES` to all four (~4 min) to
find out whether a weak gate is species-specific or a shared-template problem — each species'
record still lands in its own directory.

---

To run the same thing from a **Colab terminal** instead of the notebook:

```bash
cd /content
[ -d mesozoic-labs ] || git clone https://github.com/kuds/mesozoic-labs.git
cd mesozoic-labs

# Debian's patched setuptools breaks legacy setup.py sdists; upgrade it first.
pip install -q -U setuptools
pip install -q -e .          # core deps only — the baseline needs no torch/SB3

SPECIES=trex   # or velociraptor / brachiosaurus / dibothrosuchus
OUT=/content/drive/MyDrive/mesozoic-labs/logs/$SPECIES/zero_action_baselines
[ -d /content/drive/MyDrive ] || echo "WARNING: Drive not mounted — saving locally only"
mkdir -p "$OUT"

python environments/shared/scripts/zero_action_baseline.py "$SPECIES" \
    --episodes 40 --seed 3042 2>&1 | tee "$OUT/$(date +%Y%m%d_%H%M%S)_terminal.txt"
```

Drive is only visible to the terminal once the notebook has mounted it (cell 2 of section 1).

Pass several species names to measure them in one go — the transcript then covers all of
them, so `tee` it wherever makes sense rather than into one species' directory. Add
`--sweep-noise` to sweep `reset_noise_scale` instead of measuring a single point; that is
how the stage-1 reset noise gets calibrated.

In [ ]:
# ============================================================
# Pre-flight: zero-action baseline (stage-1 gate calibration)
# ============================================================
# Scores the do-nothing policy on stage 1 and checks each species'
# min_avg_reward gate against it.  These are measured constants: they go stale
# whenever the plant or the stage-1 reward weights change, so the result is
# saved to Drive with the plant identity and the env kwargs that produced it.

import json

from environments.shared.config import load_stage_config
from environments.shared.plant_contract import current_plant_identity
from environments.shared.scripts.zero_action_baseline import build_env, gate_margin, score

# Just the species this notebook is training (~1 min).  To find out whether a weak
# gate is species-specific or a shared-template problem, widen this to all four
# (~4 min) -- each species' record still lands in its own log directory:
#   BASELINE_SPECIES = ["velociraptor", "brachiosaurus", "dibothrosuchus", "trex"]
BASELINE_SPECIES = [SPECIES]
BASELINE_STAGE = 1
BASELINE_EPISODES = 40  # matches the figures recorded in configs/*/stage1_balance.toml
BASELINE_SEED = 3042  # the publication_evaluation seed role

results = {}
for _species in BASELINE_SPECIES:
    _env = build_env(_species, BASELINE_STAGE)
    try:
        _result = score(_env, BASELINE_EPISODES, BASELINE_SEED)
    finally:
        _env.close()

    _stage_cfg = load_stage_config(_species, BASELINE_STAGE)
    _gate = _stage_cfg["curriculum_kwargs"].get("min_avg_reward")

    # A gate only means something if it sits above the floor a statue reaches.
    if _gate is None:
        _verdict = "NO GATE"
    elif _gate <= _result["reward_mean"]:
        _verdict = "FAILS — a statue clears this gate"
    elif _result["n_standing"] == 0:
        # The statue never reaches the horizon, so there is no standing floor to
        # compare against.  That is its own problem: a plant that falls over under
        # its own home controller is not ready for a balance stage.
        _verdict = "CHECK PLANT — the statue never survives a full episode"
    elif _gate <= _result["reward_mean_standing"]:
        _verdict = "WEAK — binds only against a falling statue"
    else:
        _verdict = "OK"

    _result.update(
        {
            "species": _species,
            "stage": BASELINE_STAGE,
            "min_avg_reward": _gate,
            "verdict": _verdict,
            "margin_over_mean": gate_margin(_gate, _result["reward_mean"]),
            "margin_over_standing": gate_margin(_gate, _result["reward_mean_standing"]),
            "env_kwargs": _stage_cfg["env_kwargs"],
            "plant_identity": current_plant_identity(_species).to_dict(),
        }
    )
    results[_species] = _result

# ---- table ----------------------------------------------------------------
_header = (
    f"{'species':<16}{'reward':>10}{'mean-std':>10}{'standing':>10}"
    f"{'full-hz':>9}{'gate':>9}  verdict"
)
_lines = [
    f"zero-action baseline — stage {BASELINE_STAGE}, {BASELINE_EPISODES} episodes, seed {BASELINE_SEED}",
    "",
    _header,
    "-" * len(_header),
]
for _species, _r in results.items():
    _gate_text = "—" if _r["min_avg_reward"] is None else f"{_r['min_avg_reward']:.0f}"
    _standing_text = "—" if _r["n_standing"] == 0 else f"{_r['reward_mean_standing']:.1f}"
    _lines.append(
        f"{_species:<16}{_r['reward_mean']:>10.1f}{_r['reward_mean_minus_std']:>10.1f}"
        f"{_standing_text:>10}{_r['full_horizon_share']:>8.0%}"
        f"{_gate_text:>9}  {_r['verdict']}"
    )
_lines += [
    "",
    "A trained stage-1 policy must beat 'reward', 'mean-std' AND 'full-hz' to have",
    "learned to balance at all, and 'standing' to have learned more than 'do not fall'.",
]
_report_text = "\n".join(_lines)
print(_report_text)

# ---- save to the log directory --------------------------------------------
# One record per species, under that species' own log dir, so a baseline sits
# next to the runs it calibrates:
#   <LOG_BASE>/<species>/zero_action_baselines/<timestamp>.json
_log_base = globals().get("LOG_BASE")
if _log_base is not None:
    _stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    _captured_at = datetime.now().isoformat()

    def _payload_for(species_names):
        return {
            "schema": "mesozoic.zero-action-baseline/v1",
            "captured_at": _captured_at,
            "stage": BASELINE_STAGE,
            "episodes": BASELINE_EPISODES,
            "seed": BASELINE_SEED,
            "results": {name: results[name] for name in species_names},
        }

    for _species in results:
        _species_dir = Path(_log_base) / _species / "zero_action_baselines"
        _species_dir.mkdir(parents=True, exist_ok=True)
        (_species_dir / f"{_stamp}.json").write_text(
            json.dumps(_payload_for([_species]), indent=2, sort_keys=True, allow_nan=False)
        )
        print(f"\nSaved: {_species_dir / (_stamp + '.json')}")

    # The human-readable table covers every species measured, so it belongs with
    # the one this notebook is training.
    _table_dir = Path(_log_base) / SPECIES / "zero_action_baselines"
    _table_dir.mkdir(parents=True, exist_ok=True)
    (_table_dir / f"{_stamp}.txt").write_text(_report_text + "\n")

    # Drop a copy in the run directory too, so this run carries the calibration
    # its own gate was judged against.
    _run_dir = globals().get("RUN_DIR")
    if _run_dir is not None and SPECIES in results:
        (Path(_run_dir) / "zero_action_baseline.json").write_text(
            json.dumps(_payload_for([SPECIES]), indent=2, sort_keys=True, allow_nan=False)
        )
        print(f"Copied to run: {Path(_run_dir) / 'zero_action_baseline.json'}")
else:
    print("\nLOG_BASE not defined — run the storage-configuration cell to save results.")

## 4. Training Infrastructure

In [ ]:
import time

from stable_baselines3 import PPO, SAC

from environments.shared.evaluation import eval_policy
from environments.shared.plant_contract import validate_model_plant
from environments.shared.reporting import (
    build_stage_results_from_eval_data,
    generate_stage_artifacts,
)
from environments.shared.reporting import (
    save_evaluation_episodes as _lib_save_evaluation_episodes,
)
from environments.shared.reporting import (
    save_result_bundle as _lib_save_result_bundle,
)
from environments.shared.reporting import (
    write_training_summary as _lib_write_training_summary,
)
from environments.shared.train_base import (
    _build_core_callbacks,
    _create_or_load_model,
    _ensure_sb3,
    _maybe_ent_coef_decay_callback,
    _prepare_alg_kwargs,
    _save_final_and_sync_tb,
    _select_handoff_checkpoint,
)

ALGO_CLASS = {"PPO": PPO, "SAC": SAC}


def disconnect_runtime(reason):
    """Flush Drive writes and release the Colab runtime.

    Called whenever training halts — curriculum gate failure or normal
    completion — so an unattended "Run all" never leaves a GPU runtime
    idle. No-op outside Colab or when AUTO_DISCONNECT is False.
    """
    print(f"\n{reason}")
    if not (IN_COLAB and AUTO_DISCONNECT):
        print("Auto-disconnect skipped (not in Colab or AUTO_DISCONNECT is False).")
        return
    if USE_GOOGLE_DRIVE:
        # Flush buffered Drive writes before the runtime dies, otherwise
        # the tail of files written this session (e.g. evaluations.npz)
        # can be lost or truncated.
        from google.colab import drive

        print("Flushing Google Drive writes...")
        drive.flush_and_unmount()
    from google.colab import runtime

    print("Disconnecting runtime in 5 seconds...")
    time.sleep(5)
    runtime.unassign()


try:
    import mediapy

    _HAS_MEDIAPY = True
except ImportError:
    _HAS_MEDIAPY = False
    print("mediapy not installed. Videos will be skipped. Install with: pip install mediapy")

# Accumulates (stage_num, stage_dir) tuples as each stage completes,
# so training curves can be plotted incrementally after every stage.
completed_stages = []


def _eval_forward_vel(model, stage, vecnorm_path, n_episodes=30):
    """Evaluate a trained policy collecting forward velocity, distance, and success rate.

    Thin wrapper around the library's ``eval_policy`` that handles
    environment creation and VecNormalize loading using notebook globals.
    """
    _algo_kwargs = STAGE_CONFIGS[stage].get(f"{ALGORITHM.lower()}_kwargs", {})
    eval_env = create_vec_env(
        SPECIES_CFG,
        STAGE_CONFIGS,
        stage,
        1,
        EVALUATION_SEED,
        algorithm=ALGORITHM.lower(),
        gamma=_algo_kwargs.get("gamma"),
        plant_identity=PLANT_IDENTITY,
    )
    if vecnorm_path and Path(vecnorm_path).exists():
        load_vecnorm_stats(vecnorm_path, eval_env, current_plant=PLANT_IDENTITY)
    eval_env.training = False
    eval_env.norm_reward = False

    rewards, lengths, fwd_vels, successes, distances = eval_policy(
        model,
        eval_env,
        SPECIES_CFG.success_keys,
        n_episodes=n_episodes,
    )
    eval_env.close()
    return rewards, lengths, fwd_vels, successes, distances


def display_stage_videos(stage, stage_dir):
    """Display saved stage videos inline using mediapy.

    Finds all ``.mp4`` files in *stage_dir* and plays them inline.
    This is used after ``generate_stage_artifacts`` has already recorded
    the videos to disk.
    """
    if not _HAS_MEDIAPY:
        print(f"Skipping video display for stage {stage} (mediapy not installed).")
        return
    from environments.shared.reporting import stage_layout

    stage_dir = Path(stage_dir)
    # Resolved through stage_layout: replays live in `replays/` now, and this
    # also finds them in a legacy flat stage directory.
    mp4s = sorted(p for p in stage_layout.iter_replay_files(stage_dir) if p.suffix == ".mp4")
    if not mp4s:
        print(f"No videos found in {stage_dir}")
        return
    for mp4 in mp4s:
        print(f"Playing: {mp4.name}")
        mediapy.show_video(mediapy.read_video(str(mp4)), fps=50)


def train_stage(
    stage,
    timesteps,
    load_path=None,
    run_dir=None,
    vecnorm_path=None,
):
    """Train a single curriculum stage.

    Delegates algorithm setup, model creation, callback construction,
    and checkpoint saving to shared helpers in ``train_base.py``.
    Notebook-specific concerns (interactive printing, evaluation,
    curriculum gating) are handled here.

    Returns (model, best_model_path, final_model_path, stage_dir, vecnorm_save_path, stage_results).
    """
    stage_start = time.time()

    config = STAGE_CONFIGS[stage]
    sb3 = _ensure_sb3()
    AlgoClass = ALGO_CLASS[ALGORITHM.upper()]

    # Directories — each stage gets a subdirectory within the run directory
    if run_dir is None:
        run_dir = repo_root / "logs"
    stage_dir = Path(run_dir) / f"stage{stage}"
    stage_dir.mkdir(parents=True, exist_ok=True)
    model_dir = stage_dir / "models"
    model_dir.mkdir(exist_ok=True)

    # Resolve stage transition parameters up front so the exact values
    # (whether from TOML or defaults) are recorded in the saved config.
    cur_kwargs = config.get("curriculum_kwargs", {})
    extra_meta = {"seed": SEED, "n_envs": N_ENVS, "timesteps": timesteps}
    if stage > 1:
        warmup_timesteps = cur_kwargs.get("warmup_timesteps", 100_000)
        warmup_clip_range = cur_kwargs.get("warmup_clip_range", 0.02)
        warmup_ent_coef = cur_kwargs.get("warmup_ent_coef", 0.02)
        ramp_timesteps = cur_kwargs.get("ramp_timesteps", 500_000)
        ramp_start_value = cur_kwargs.get("ramp_start_value", 0.1)
        target_fwd_weight = config["env_kwargs"].get("forward_vel_weight", 1.0)
        extra_meta.update(
            {
                "warmup_timesteps": warmup_timesteps,
                "warmup_clip_range": warmup_clip_range,
                "warmup_ent_coef": warmup_ent_coef,
                "ramp_timesteps": ramp_timesteps,
                "ramp_start_value": ramp_start_value,
                "forward_vel_weight": target_fwd_weight,
            }
        )

    print(f"{'=' * 60}")
    print(f"Stage {stage}: {config['name']} ({ALGORITHM})")
    print(f"Description: {config['description']}")
    print(f"Timesteps: {timesteps:,}")
    print(f"Log dir: {stage_dir}")
    if vecnorm_path:
        print(f"VecNormalize from prior stage: {vecnorm_path}")
    print(f"{'=' * 60}")

    # Save reward weights, hyperparameters, and resolved stage transition
    # params for reproducibility.
    cfg_path = save_stage_config(
        stage_dir,
        stage,
        config,
        ALGORITHM,
        extra=extra_meta,
        env_class=EnvClass,
        species=SPECIES,
        plant_identity=PLANT_IDENTITY,
    )
    print(f"Stage config saved to: {cfg_path}")

    # Create environments using library infrastructure.
    use_subproc = ALGORITHM.lower() == "sac" and N_ENVS > 1
    if use_subproc:
        print(f"Using SubprocVecEnv for SAC ({N_ENVS} parallel workers)")
    _algo_kwargs_cfg = config.get(f"{ALGORITHM.lower()}_kwargs", {})
    _alg_gamma = _algo_kwargs_cfg.get("gamma")
    train_env = create_vec_env(
        SPECIES_CFG,
        STAGE_CONFIGS,
        stage,
        N_ENVS,
        SEED,
        use_subproc=use_subproc,
        algorithm=ALGORITHM.lower(),
        gamma=_alg_gamma,
        plant_identity=PLANT_IDENTITY,
    )
    eval_env = create_vec_env(
        SPECIES_CFG,
        STAGE_CONFIGS,
        stage,
        1,
        CHECKPOINT_SELECTION_SEED,
        algorithm=ALGORITHM.lower(),
        gamma=_alg_gamma,
        plant_identity=PLANT_IDENTITY,
    )

    # Load VecNormalize stats from prior stage
    if vecnorm_path:
        if not load_vecnorm_stats(vecnorm_path, train_env, eval_env, current_plant=PLANT_IDENTITY):
            print(f"WARNING: VecNormalize file not found: {vecnorm_path} — eval env will use defaults")
            eval_env.training = False
            eval_env.norm_reward = False
    else:
        eval_env.training = False
        eval_env.norm_reward = False

    # Build algorithm kwargs using shared helper (handles LR schedules,
    # clip range annealing, and TensorBoard setup).
    alg_kwargs, local_tb_dir, gcs_tb_path = _prepare_alg_kwargs(
        config,
        ALGORITHM.lower(),
        VERBOSE,
        stage_dir,
        use_tensorboard=True,
    )

    # Create or load model using shared helper
    if load_path:
        print(f"Loading model from: {load_path}")
    model = _create_or_load_model(
        sb3,
        ALGORITHM.lower(),
        alg_kwargs,
        train_env,
        load_path,
        plant_identity=PLANT_IDENTITY,
    )

    # Build callbacks using shared helper (includes EvalCallback,
    # CheckpointCallback, DiagnosticsCallback, and EvalCollapseEarlyStopCallback).
    callbacks, eval_callback, save_vecnorm_cb = _build_core_callbacks(
        sb3,
        eval_env,
        model_dir,
        stage_dir,
        stage,
        N_ENVS,
        eval_freq=50000,
        save_freq=500000,
        verbose=VERBOSE,
        stage_config=config,
        species=SPECIES,
    )
    best_vecnorm_path = save_vecnorm_cb.save_path

    # Optional PPO entropy-coefficient decay (set ent_coef_end in the TOML).
    ent_decay_cb = _maybe_ent_coef_decay_callback(config, ALGORITHM.lower(), timesteps)
    if ent_decay_cb is not None:
        callbacks.append(ent_decay_cb)

    # Stage transition callbacks (stages 2+): warm up the value function
    # and gradually ramp the forward velocity reward to prevent
    # catastrophic forgetting of balance behaviours.
    if stage > 1 and ALGORITHM.upper() == "PPO":
        callbacks.append(
            StageWarmupCallback(
                warmup_timesteps=warmup_timesteps,
                warmup_clip_range=warmup_clip_range,
                warmup_ent_coef=warmup_ent_coef,
            )
        )
    if stage > 1:
        callbacks.append(
            RewardRampCallback(
                attr_name="forward_vel_weight",
                start_value=ramp_start_value,
                end_value=target_fwd_weight,
                ramp_timesteps=ramp_timesteps,
            )
        )

    # Train
    model.learn(
        total_timesteps=timesteps,
        callback=sb3["CallbackList"](callbacks),
        progress_bar=VERBOSE >= 1,
    )

    # Actual steps trained — differs from the configured budget when a
    # callback (e.g. EvalCollapseEarlyStopCallback) ended training early.
    actual_timesteps = int(model.num_timesteps)
    if actual_timesteps < timesteps:
        print(f"\nNOTE: training stopped early at {actual_timesteps:,} of {timesteps:,} timesteps.")

    # Save final model and sync TensorBoard using shared helper
    final_path = _save_final_and_sync_tb(
        model,
        train_env,
        model_dir,
        stage,
        local_tb_dir,
        gcs_tb_path,
    )
    final_vecnorm_path = str(final_path) + "_vecnorm.pkl"
    vecnorm_save_path = final_vecnorm_path
    print(f"\nFinal model saved to: {final_path}.zip")
    print(f"VecNormalize stats saved to: {final_vecnorm_path}")

    # Evaluate the final model with its matching VecNormalize stats.
    episode_rewards, episode_lengths, episode_fwd_vels, episode_successes, episode_distances = _eval_forward_vel(
        model,
        stage,
        final_vecnorm_path,
        n_episodes=30,
    )
    final_eval_path = _lib_save_evaluation_episodes(
        stage_dir,
        rewards=episode_rewards,
        lengths=episode_lengths,
        forward_velocities=episode_fwd_vels,
        distances=episode_distances,
        successes=episode_successes,
        evaluation_seed=EVALUATION_SEED,
        checkpoint_label="final",
    )
    print(f"Final-model episode evidence saved to: {final_eval_path}")
    mean_reward = float(np.mean(episode_rewards))
    std_reward = float(np.std(episode_rewards))
    mean_length = float(np.mean(episode_lengths))
    std_length = float(np.std(episode_lengths))
    mean_fwd_vel = float(np.mean(episode_fwd_vels))
    std_fwd_vel = float(np.std(episode_fwd_vels))
    mean_distance = float(np.mean(episode_distances))
    sim_dt = float(eval_env.get_attr("dt")[0])
    mean_success_rate = float(np.mean(episode_successes))
    print(f"Eval (final model): mean_reward={mean_reward:.2f} +/- {std_reward:.2f}")
    print(f"Eval: mean_length={mean_length:.1f} +/- {std_length:.1f} steps ({mean_length * sim_dt:.2f}s sim time)")
    print(f"Eval: mean_forward_vel={mean_fwd_vel:.2f} +/- {std_fwd_vel:.2f} m/s")
    print(f"Eval: mean_distance_traveled={mean_distance:.2f} m")
    print(f"Eval: success_rate={mean_success_rate:.0%}")

    train_env.close()
    eval_env.close()

    # Build base results dict from on-disk eval data (evaluations.npz),
    # then enrich with the live 30-episode evaluation metrics above.
    stage_duration = time.time() - stage_start
    stage_results = build_stage_results_from_eval_data(
        stage_dir,
        stage,
        config,
        timesteps=actual_timesteps,
        duration_seconds=stage_duration,
    )
    best_eval_reward = stage_results["best_eval_reward"]
    best_eval_std = stage_results["best_eval_std"]
    best_eval_length = stage_results["best_eval_length"]
    best_eval_timestep = stage_results["best_eval_timestep"]
    if best_eval_reward != "":
        print(f"Best model eval:  mean_reward={best_eval_reward} +/- {best_eval_std} (at {best_eval_timestep:,} steps)")

    # Override with richer live-eval metrics
    stage_results.update(
        {
            "mean_reward": mean_reward,
            "std_reward": std_reward,
            "mean_episode_length": mean_length,
            "std_episode_length": std_length,
            "mean_forward_vel": mean_fwd_vel,
            "std_forward_vel": std_fwd_vel,
            "mean_distance_traveled": mean_distance,
            "mean_success_rate": mean_success_rate,
            "sim_dt": sim_dt,
        }
    )

    # The SELECTED checkpoint: next-stage loading, the evidence CSV below, the
    # replay video, and the stance gate report must all describe the same
    # policy. `_select_handoff_checkpoint` is the one selector they share --
    # this cell used to carry a private copy of the preference order, which is
    # how the replay ended up showing `best_model` while
    # `evaluation_selected.csv` was evidence for `robust_best_model`.
    #
    # It prefers the risk-adjusted robust_best_model (highest mean - std eval)
    # over SB3's mean-reward best_model: a high mean can be propped up by a good
    # mode while a fat failure tail is already growing (run 20260709_185946).
    # It requires the matched VecNormalize stats, so it cannot return a hybrid.
    _handoff = _select_handoff_checkpoint(model_dir)
    if _handoff is None:
        best_model_zip = model_dir / "best_model.zip"
        best_vecnorm_candidate = best_vecnorm_path
        if best_model_zip.exists():
            raise FileNotFoundError(
                f"Selected checkpoint {best_model_zip} is missing its matched VecNormalize state "
                f"{best_vecnorm_candidate}; refusing to evaluate or export a hybrid checkpoint."
            )
    else:
        _selected_name, _selected_path, best_vecnorm_candidate = _handoff
        best_model_zip = model_dir / f"{_selected_name}.zip"
        print(f"Selected checkpoint: {_selected_name}")
    best_model_reward, best_model_std_reward = "", ""
    best_model_length, best_model_std_length = "", ""
    best_model_fwd_vel, best_model_std_fwd_vel = "", ""
    best_model_distance = ""
    best_model_success_rate = ""
    if best_model_zip.exists():
        best_path = model_dir / best_model_zip.stem
        model = AlgoClass.load(str(best_path))
        validate_model_plant(model, PLANT_IDENTITY, artifact=str(best_model_zip))
        output_path = str(best_path)
        print(f"Loaded best model for next-stage: {best_path}.zip")

        # Guaranteed by _select_handoff_checkpoint, which only returns a
        # candidate whose matched statistics exist. Kept as an assertion
        # because exporting a hybrid checkpoint is silent and unrecoverable.
        if not Path(best_vecnorm_candidate).exists():
            raise FileNotFoundError(
                f"Selected checkpoint {best_model_zip} is missing its matched VecNormalize state "
                f"{best_vecnorm_candidate}; refusing to evaluate or export a hybrid checkpoint."
            )
        vecnorm_save_path = best_vecnorm_candidate
        print(f"Using matched VecNormalize for selected checkpoint: {vecnorm_save_path}")

        # Evaluate the best model over 30 episodes
        print("Evaluating best model (30 episodes)...")
        bm_rewards, bm_lengths, bm_fwd_vels, bm_successes, bm_distances = _eval_forward_vel(
            model,
            stage,
            vecnorm_save_path,
            n_episodes=30,
        )
        selected_eval_path = _lib_save_evaluation_episodes(
            stage_dir,
            rewards=bm_rewards,
            lengths=bm_lengths,
            forward_velocities=bm_fwd_vels,
            distances=bm_distances,
            successes=bm_successes,
            evaluation_seed=EVALUATION_SEED,
            checkpoint_label="selected",
        )
        print(f"Selected-model episode evidence saved to: {selected_eval_path}")
        best_model_reward = round(float(np.mean(bm_rewards)), 2)
        best_model_std_reward = round(float(np.std(bm_rewards)), 2)
        best_model_length = round(float(np.mean(bm_lengths)), 1)
        best_model_std_length = round(float(np.std(bm_lengths)), 1)
        best_model_fwd_vel = round(float(np.mean(bm_fwd_vels)), 2)
        best_model_std_fwd_vel = round(float(np.std(bm_fwd_vels)), 2)
        best_model_distance = round(float(np.mean(bm_distances)), 2)
        best_model_success_rate = round(float(np.mean(bm_successes)), 2)
        print(f"Best model eval:  mean_reward={best_model_reward} +/- {best_model_std_reward}")
        print(f"Best model eval:  mean_length={best_model_length} +/- {best_model_std_length}")
        print(f"Best model eval:  mean_fwd_vel={best_model_fwd_vel} +/- {best_model_std_fwd_vel} m/s")
        print(f"Best model eval:  mean_distance={best_model_distance} m")
        print(f"Best model eval:  success_rate={best_model_success_rate:.0%}")
    else:
        output_path = str(final_path)
        selected_eval_path = _lib_save_evaluation_episodes(
            stage_dir,
            rewards=episode_rewards,
            lengths=episode_lengths,
            forward_velocities=episode_fwd_vels,
            distances=episode_distances,
            successes=episode_successes,
            evaluation_seed=EVALUATION_SEED,
            checkpoint_label="selected",
        )
        print(f"Selected-model episode evidence saved to: {selected_eval_path}")
        best_model_reward = round(float(np.mean(episode_rewards)), 2)
        best_model_std_reward = round(float(np.std(episode_rewards)), 2)
        best_model_length = round(float(np.mean(episode_lengths)), 1)
        best_model_std_length = round(float(np.std(episode_lengths)), 1)
        best_model_fwd_vel = round(float(np.mean(episode_fwd_vels)), 3)
        best_model_std_fwd_vel = round(float(np.std(episode_fwd_vels)), 3)
        best_model_distance = round(float(np.mean(episode_distances)), 3)
        best_model_success_rate = round(float(np.mean(episode_successes)), 4)

    # The curriculum gate is NOT evaluated here. `generate_stage_artifacts`
    # evaluates it, through the one shared `reporting.gates.evaluate_stage_gate`,
    # and records `gate_passed` / `publication_gate_passed` / `gate_failures`
    # onto this dict; the per-stage cells below enforce it.
    #
    # This cell used to carry its own checklist over min_avg_reward /
    # min_avg_episode_length / min_avg_forward_vel / min_success_rate. It knew
    # nothing about `gate_kind`, so when T-Rex stage 1 moved to
    # stance_quality/v1 and retired min_avg_episode_length, the checklist
    # quietly degraded to a reward comparison alone -- which the zero-action
    # statue clears by 68%. Run 20260802_203215 recorded
    # publication_gate_passed = True beside a stance_gate_report.txt reading
    # GATE: FAIL at 10.6x the duty ceiling, and advanced to stage 2 on it.
    # Keep the rule in the library where both backends and the sweep worker
    # read the same copy.

    # Update stage_results with model paths and best-model eval
    stage_results.update(
        {
            "model_path": output_path,
            "final_model_path": str(final_path),
            "vecnorm_path": vecnorm_save_path,
            "final_vecnorm_path": final_vecnorm_path,
            "best_model_reward": best_model_reward,
            "best_model_std_reward": best_model_std_reward,
            "best_model_length": best_model_length,
            "best_model_std_length": best_model_std_length,
            "best_model_fwd_vel": best_model_fwd_vel,
            "best_model_std_fwd_vel": best_model_std_fwd_vel,
            "best_model_distance": best_model_distance,
            "best_model_success_rate": best_model_success_rate,
        }
    )

    return model, output_path, str(final_path), stage_dir, vecnorm_save_path, stage_results


def write_training_summary(run_dir, stage_results_list, species=None):
    """Write a training summary text file to the run directory."""
    if species is None:
        species = SPECIES
    path = _lib_write_training_summary(
        run_dir,
        stage_results_list,
        species=species,
        algorithm=ALGORITHM,
        seed=SEED,
        n_envs=N_ENVS,
        quick_test=QUICK_TEST,
    )
    summary_text = path.read_text()
    print(f"\nTraining summary saved to: {path}")
    print(summary_text)


def save_run_bundle(stage_results_list, run_dir=None, species=SPECIES):
    """Regenerate the canonical, Drive-portable bundle for completed stages."""
    if run_dir is None:
        run_dir = RUN_DIR
    paths = _lib_save_result_bundle(
        stage_results_list,
        stage_configs=STAGE_CONFIGS,
        species=species,
        algorithm=ALGORITHM,
        seed=SEED,
        run_dir=run_dir,
        backend="stable-baselines3",
        hardware=HARDWARE_LABEL,
        parallel_envs=N_ENVS,
        evaluation_episodes=30,
        evaluation_seeds=[CHECKPOINT_SELECTION_SEED, EVALUATION_SEED],
        seed_roles={
            "training": SEED,
            "checkpoint_selection_evaluation": CHECKPOINT_SELECTION_SEED,
            "publication_evaluation": EVALUATION_SEED,
        },
        plant_identity=PLANT_IDENTITY.to_dict(),
        run_id=RUN_ID,
        repository_root=repo_root,
    )
    print("\nResult bundle updated:")
    for name, path in paths.items():
        print(f"  {name}: {path}")
    return paths


print(f"Training infrastructure ready. Algorithm: {ALGORITHM}")
print("DiagnosticsCallback enabled: per-component rewards, obs/action stats,")
print("VecNormalize tracking, termination reasons, and plateau detection")
print("will log to TensorBoard.")

### Visualization Functions

In [ ]:
from environments.shared.visualization import (
    plot_diagnostics_graphs as _lib_plot_diagnostics_graphs,
)
from environments.shared.visualization import (
    plot_training_curves as _lib_plot_training_curves,
)


def plot_training_curves(stage_dirs, stage_configs, algo_name, save_path=None):
    """Plot evaluation reward, episode length, tilt angle, and forward velocity curves.

    Thin wrapper around the shared library function that fills in the
    notebook's SPECIES global and calls ``plt.show()`` for inline display.
    """
    _lib_plot_training_curves(
        stage_dirs,
        stage_configs,
        species=SPECIES,
        algorithm=algo_name,
        save_path=save_path,
    )
    if save_path is not None:
        print(f"Training curves saved to: {save_path}")
    plt.show()


def plot_diagnostics_graphs(stage_dirs, stage_configs, algo_name, save_dir=None, _show=True):
    """Create diagnostic figures for locomotion health and behavioral metrics.

    Thin wrapper around the shared library function that fills in the
    notebook's SPECIES global.  When ``_show`` is True (default), figures
    are displayed inline; otherwise they are closed after saving.
    """
    fig1, fig2 = _lib_plot_diagnostics_graphs(
        stage_dirs,
        stage_configs,
        species=SPECIES,
        algorithm=algo_name,
        save_dir=save_dir,
        show=_show,
    )
    if _show:
        plt.show()


print("Visualization functions ready (shared library).")

## 5. Stage 1: Balance

The selected agent learns to stand upright without falling. No forward velocity reward —
just a strong alive bonus.

In [ ]:
cur1 = STAGE_CONFIGS[1]["curriculum_kwargs"]
timesteps_1 = 50_000 if QUICK_TEST else cur1["timesteps"]

model_1, path_1, final_path_1, dir_1, vecnorm_1, results_1 = train_stage(
    stage=1, timesteps=timesteps_1, run_dir=RUN_DIR
)

In [ ]:
# Generate all stage artifacts: summary, videos, and graphs (saved to disk).
# This also evaluates the stage's declared curriculum gate and writes the
# verdict into results_1 (gate_passed / publication_gate_passed /
# gate_failures), which the check at the bottom of this cell enforces.
results_1 = generate_stage_artifacts(
    species_cfg=SPECIES_CFG,
    stage_config=STAGE_CONFIGS[1],
    stage=1,
    algorithm=ALGORITHM,
    stage_dir=dir_1,
    seed=SEED,
    stage_results=results_1,
)

# Display saved videos inline
display_stage_videos(1, dir_1)

# Display training curves and diagnostics inline
completed_stages.append((1, dir_1))
plot_training_curves([(1, dir_1)], STAGE_CONFIGS, ALGORITHM)
plot_diagnostics_graphs([(1, dir_1)], STAGE_CONFIGS, ALGORITHM)

# Write training summary with all completed stages so far (before gate check
# so the summary is always generated even if the gate fails)
write_training_summary(RUN_DIR, [results_1])

# Regenerate the partial canonical bundle after all stage artifacts exist.
bundle_paths_1 = save_run_bundle([results_1], species=SPECIES)

# Enforce curriculum gate after generating artifacts. On failure,
# release the runtime first: the raise below halts "Run all", so the
# auto-disconnect cell at the end would never execute.
if not results_1["publication_gate_passed"]:
    _gate_msg = "Stage 1 failed curriculum gate: " + "; ".join(results_1["gate_failures"]) + "."
    disconnect_runtime(_gate_msg)
    raise RuntimeError(_gate_msg)

## 6. Stage 2: Locomotion

Starting from the Stage 1 checkpoint, the selected agent learns to move forward.

In [ ]:
cur2 = STAGE_CONFIGS[2]["curriculum_kwargs"]
timesteps_2 = 50_000 if QUICK_TEST else cur2["timesteps"]

model_2, path_2, final_path_2, dir_2, vecnorm_2, results_2 = train_stage(
    stage=2, timesteps=timesteps_2, load_path=path_1, run_dir=RUN_DIR, vecnorm_path=vecnorm_1
)

In [ ]:
# Generate all stage artifacts: summary, videos, and graphs (saved to disk).
# This also evaluates the stage's declared curriculum gate and writes the
# verdict into results_2 (gate_passed / publication_gate_passed /
# gate_failures), which the check at the bottom of this cell enforces.
results_2 = generate_stage_artifacts(
    species_cfg=SPECIES_CFG,
    stage_config=STAGE_CONFIGS[2],
    stage=2,
    algorithm=ALGORITHM,
    stage_dir=dir_2,
    seed=SEED,
    stage_results=results_2,
)

# Display saved videos inline
display_stage_videos(2, dir_2)

# Display training curves and diagnostics inline
completed_stages.append((2, dir_2))
plot_training_curves([(2, dir_2)], STAGE_CONFIGS, ALGORITHM)
plot_diagnostics_graphs([(2, dir_2)], STAGE_CONFIGS, ALGORITHM)

# Write training summary with all completed stages so far (before gate check
# so the summary is always generated even if the gate fails)
write_training_summary(RUN_DIR, [results_1, results_2])

# Regenerate the partial canonical bundle after all stage artifacts exist.
bundle_paths_2 = save_run_bundle([results_1, results_2], species=SPECIES)

# Enforce curriculum gate after generating artifacts. On failure,
# release the runtime first: the raise below halts "Run all", so the
# auto-disconnect cell at the end would never execute.
if not results_2["publication_gate_passed"]:
    _gate_msg = "Stage 2 failed curriculum gate: " + "; ".join(results_2["gate_failures"]) + "."
    disconnect_runtime(_gate_msg)
    raise RuntimeError(_gate_msg)

## 7. Stage 3: Species-Specific Behavior

The final stage enables the species-specific reward (strike/bite/food reach/snap) and trains the full behavioral repertoire. The stage 3 label and description are configured in the species selection cell above.

In [ ]:
cur3 = STAGE_CONFIGS[3]["curriculum_kwargs"]
timesteps_3 = 50_000 if QUICK_TEST else cur3["timesteps"]

model_3, path_3, final_path_3, dir_3, vecnorm_3, results_3 = train_stage(
    stage=3, timesteps=timesteps_3, load_path=path_2, run_dir=RUN_DIR, vecnorm_path=vecnorm_2
)

In [ ]:
# Generate all stage artifacts: summary, videos, and graphs (saved to disk).
# This also evaluates the stage's declared curriculum gate and writes the
# verdict into results_3 (gate_passed / publication_gate_passed /
# gate_failures), which the check at the bottom of this cell enforces.
results_3 = generate_stage_artifacts(
    species_cfg=SPECIES_CFG,
    stage_config=STAGE_CONFIGS[3],
    stage=3,
    algorithm=ALGORITHM,
    stage_dir=dir_3,
    seed=SEED,
    stage_results=results_3,
)

# Display saved videos inline
display_stage_videos(3, dir_3)

# Display training curves and diagnostics inline
completed_stages.append((3, dir_3))
plot_training_curves([(3, dir_3)], STAGE_CONFIGS, ALGORITHM)
plot_diagnostics_graphs([(3, dir_3)], STAGE_CONFIGS, ALGORITHM)

# Write training summary with all completed stages so far (before gate check
# so the summary is always generated even if the gate fails)
write_training_summary(RUN_DIR, [results_1, results_2, results_3])

# Generate the complete schema-v2 summary and hash manifest.
bundle_paths_3 = save_run_bundle([results_1, results_2, results_3], species=SPECIES)

# Enforce curriculum gate after generating artifacts. On failure,
# release the runtime first: the raise below halts "Run all", so the
# auto-disconnect cell at the end would never execute.
if not results_3["publication_gate_passed"]:
    _gate_msg = "Stage 3 failed curriculum gate: " + "; ".join(results_3["gate_failures"]) + "."
    disconnect_runtime(_gate_msg)
    raise RuntimeError(_gate_msg)

## 8. Evaluate Final Policy

In [ ]:
# Evaluate the final Stage 3 policy using the library's evaluate() function,
# which provides full locomotion metrics (gait symmetry, cost of transport,
# stride frequency, velocity consistency, etc.) via LocomotionMetrics.
print(f"Evaluating final Stage 3 policy ({ALGORITHM})...")
evaluate(
    species_cfg=SPECIES_CFG,
    stage_configs=STAGE_CONFIGS,
    model_path=path_3 + ".zip",
    n_episodes=30,
    render=False,
    stage=3,
    algorithm=ALGORITHM.lower(),
)

## 9. Training Curves

In [ ]:
# Re-plot individual stage graphs (convenient for re-running this cell standalone)
for stage_num, stage_dir in completed_stages:
    print(f"\n{'=' * 40}")
    print(f"Stage {stage_num}: {STAGE_CONFIGS[stage_num]['name']}")
    print(f"{'=' * 40}")
    plot_training_curves(
        [(stage_num, stage_dir)], STAGE_CONFIGS, ALGORITHM, save_path=Path(stage_dir) / "training_curves.png"
    )
    plot_diagnostics_graphs([(stage_num, stage_dir)], STAGE_CONFIGS, ALGORITHM, save_dir=stage_dir)

## 10. Replay All Stage Videos

Display the saved videos for all completed training stages. Videos are recorded
by `generate_stage_artifacts` after each stage completes.

In [ ]:
for stage_num, stage_dir in completed_stages:
    print(f"\n{'=' * 40}")
    print(f"Stage {stage_num}: {STAGE_CONFIGS[stage_num]['name']}")
    print(f"{'=' * 40}")
    display_stage_videos(stage_num, stage_dir)

## 11. Cleanup

In [ ]:
# Verify the complete bundle without rewriting immutable artifacts.
bundle_report = validate_result_bundle(RUN_DIR, require_complete=True)
print(f"Result bundle verified: {bundle_report['status']}")

print("Training complete!")
print(f"\nAlgorithm: {ALGORITHM}")
print(f"Run directory: {RUN_DIR}")
print(f"Stage 1 best model: {path_1}.zip")
print(f"Stage 1 final model: {final_path_1}.zip")
print(f"Stage 2 best model: {path_2}.zip")
print(f"Stage 2 final model: {final_path_2}.zip")
print(f"Stage 3 best model: {path_3}.zip")
print(f"Stage 3 final model: {final_path_3}.zip")
print("\nTo run the other algorithm, change ALGORITHM at the top and re-run all cells.")

## 12. Auto-Disconnect

Releases the Colab runtime when the full curriculum finishes so it doesn't sit idle burning GPU credits. Stages that fail their curriculum gate disconnect the same way from within their gate check (the gate's `RuntimeError` halts "Run all" before this cell). Set `AUTO_DISCONNECT = False` in the configuration cell to keep the runtime alive for interactive work.

In [ ]:
disconnect_runtime("Training finished — full curriculum complete.")